# Job A val_defect - Feature-based Real-IAD benchmark (Colab)

Runs PatchCore + PaDiM + SubspaceAD on the pending Real-IAD val_defect categories relative to the full 30-category dataset, camera C1 only.

**Before running:**
- Runtime -> Change runtime type -> GPU (T4/V100/A100).
- `Real-IAD_dataset/realiad_1024/*.zip` must be on Drive (zipped, one per category).
- The notebook uses `colab_featurebased_val_defect.yaml`, which switches thresholding to `val_f1` and relies on the patched splitter that routes 10% of anomaly images into val.

**Outputs:** synced to `/content/drive/MyDrive/thesis_runs/jobA_val_defect/` per-category. Safe to interrupt and restart - finished categories get a `<category>.done` marker and are skipped on resume.

In [ ]:
# 1. Mount Drive and verify the pending val_defect category zips are visible.
from google.colab import drive
drive.mount('/content/drive')

import os, glob
ZIPS_DIR = '/content/drive/MyDrive/Real-IAD_dataset/realiad_1024'
CATEGORIES = [
    'audiojack', 'bottle_cap', 'button_battery', 'end_cap', 'eraser',
    'fire_hood', 'mint', 'mounts', 'pcb', 'phone_battery',
    'plastic_nut', 'plastic_plug', 'porcelain_doll', 'regulator', 'tape',
    'toy', 'toy_brick', 'u_block', 'usb', 'usb_adaptor',
    'vcpill', 'wooden_beads', 'woodstick',
]
missing = [c for c in CATEGORIES if not os.path.isfile(os.path.join(ZIPS_DIR, f'{c}.zip'))]
print('zip dir:', ZIPS_DIR)
print('selected categories:', ', '.join(CATEGORIES))
print('zip files visible:', len(glob.glob(os.path.join(ZIPS_DIR, '*.zip'))))
assert not missing, f'Missing required zips: {missing}'
print('All required category archives are present.')

In [ ]:
# 2. Clone (or pull) the thesis repo into /content/.
# Replace <your-username>/<your-repo> with the repo you pushed to.
REPO_URL = 'https://github.com/LorenzSF/Real-time-visual-defect-detection.git'
REPO_DIR = '/content/Real-time-visual-defect-detection'
BRANCH   = 'main'

import os, subprocess
if os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', '-C', REPO_DIR, 'fetch', '--all'])
    subprocess.check_call(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])

print('Repo ready at', REPO_DIR)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'log', '-1', '--oneline']).decode().strip())

In [ ]:
# 3. Install Python dependencies for the three models.
# Anomalib adapters require anomalib >= 2.x (module path `anomalib.models.image.*`).
# transformers stays on the 4.x branch for compatibility with anomalib 2.x and DINOv2-with-registers.
!pip install -q "anomalib>=2.2,<3" lightning "transformers>=4.46,<5" scikit-learn timm kornia einops
!pip install -q -e /content/Real-time-visual-defect-detection --no-deps || echo '[note] editable install skipped (not required)'

import torch, anomalib, lightning, transformers
print('torch       ', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')
print('anomalib    ', anomalib.__version__)
print('lightning   ', lightning.__version__)
print('transformers', transformers.__version__)

In [ ]:
# 4. Launch the val_defect category loop. Streams logs live; safe to interrupt and rerun.
import os
os.environ['ZIPS_DIR']    = '/content/drive/MyDrive/Real-IAD_dataset/realiad_1024'
os.environ['RESULTS_DIR'] = '/content/drive/MyDrive/thesis_runs/jobA_val_defect'
os.environ['WORK_DIR']    = '/content/work'
os.environ['REPO_DIR']    = '/content/Real-time-visual-defect-detection'
os.environ['CONFIG']      = '/content/Real-time-visual-defect-detection/src/benchmark_AD/configs/colab_featurebased_val_defect.yaml'

# Ensure the Drive log dir exists before `tee` tries to append to it.
!mkdir -p {os.environ['RESULTS_DIR']}
!bash /content/Real-time-visual-defect-detection/scripts/run_jobA_val_defect_colab.sh 2>&1 | tee -a {os.environ['RESULTS_DIR']}/run.log

## Troubleshooting

- **`Missing required zips`**: one of the selected pending categories is not present under `Real-IAD_dataset/realiad_1024/`. Upload or rename the missing archive and rerun cell 1.
- **`No Real-IAD images found`**: the zip did not unpack with the expected `OK/` + `NG/<defect>/S000X/` layout. Inspect one extracted category in `/content/work/<category>/` and share the `ls` output.
- **`CUDA out of memory` on SubspaceAD**: lower `subspacead.batch_size` in `colab_featurebased_val_defect.yaml` from 4 to 2.
- **Session disconnected mid-run**: reopen the notebook, rerun cells 1-4 - completed categories are skipped via `.done` markers.
- **Drive I/O stalls**: check `Runtime -> Manage sessions`; if Drive is throttled, wait a few minutes or reconnect.

In [ ]:
# 6. Release Colab resources once the run is finished and synced.
# This clears local caches first, then disconnects the runtime.
import gc

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

gc.collect()

try:
    from google.colab import runtime
    runtime.unassign()
except Exception:
    pass
